# Training Notebook

In [6]:
import sys
import os

# Add the parent directory to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from ml_attack import get_no_mod, LWEDataset, get_filename_from_params
from ml_attack.utils import get_continuous_reduction_default_params, get_default_params, get_percentage_true_b, get_train_default_params, cmod, mod_mult

import numpy as np

from scipy.linalg import circulant
from ml_attack.lwe import neg_circ

from ml_attack.continuous_reduction import ContinuousReduction
from concurrent.futures import ProcessPoolExecutor

from itertools import product
np.set_printoptions(linewidth=np.inf, threshold=np.inf)


## Dataset creation

Training debug:

In [ ]:
params = get_default_params()
params.update(get_continuous_reduction_default_params())
params.update(get_train_default_params())
params.update({
    'n': 64,
    'q': 3329,
    'k': 2,
    'eta': 2,
    'secret_type': 'cbd',
    'error_type': 'cbd',

    'num_gen': 4,
    'seed': 42,

    'num_matrices': -1,
    'reduction_max_size': 20,
    'float_type': 'd',
    'matrix_config': 'dual',
    'interleaved_steps': 0,
    'reduction_samples': -1,
    'reduction_resampling': False,
    'warmup_steps': 4,
    'bkz_block_sizes': "20:40:10",
    
    'penalty': 3,
    'verbose': True,
    'continuous_reduction': True,

    "train_percentages": [0.1, 0.3, 0.6, 1]
})

filename = get_filename_from_params(params)

reload = False
if os.path.exists(filename) and reload:
    print(f"Loading dataset from {filename}")
    dataset = LWEDataset.load_reduced(filename)
    params = dataset.params
    dataset.approximate_b()
else:
    print(f"Generating dataset and saving to {filename}")
    dataset = LWEDataset(params)
    dataset.initialize()
    dataset.attack(
        stop_strategy="hour",
        stop_after=4,
        attack_strategy="minute",
        attack_every=30,
        save_at_the_end=True
    )

Generating dataset and saving to ./data_n_64_k_2_s_cbd_4a6fb.pkl
Samples coverage: 1.0000
Samples reuse: 2.1875
Attacking 8 matrices using 8 threads.
- Algo: flatter | Updated 20/20 (268 non zero) | Mean std_B: 9998.73 | Min row norm: 10479.10 | Min col norm: 4985.85
- Algo: flatter | Updated 20/20 (268 non zero) | Mean std_B: 9732.02 | Min row norm: 9343.59 | Min col norm: 4871.75
- Algo: flatter | Updated 20/20 (268 non zero) | Mean std_B: 9799.52 | Min row norm: 9306.37 | Min col norm: 5712.44
- Algo: flatter | Updated 20/20 (268 non zero) | Mean std_B: 9980.81 | Min row norm: 10182.95 | Min col norm: 5860.20
- Algo: flatter | Updated 20/20 (268 non zero) | Mean std_B: 9756.25 | Min row norm: 10239.98 | Min col norm: 5417.59
- Algo: flatter | Updated 20/20 (266 non zero) | Mean std_B: 9860.02 | Min row norm: 10428.51 | Min col norm: 5659.67
- Algo: flatter | Updated 20/20 (267 non zero) | Mean std_B: 9964.97 | Min row norm: 10329.69 | Min col norm: 5064.61
- Algo: flatter | Updated 

In [4]:
dataset.approximate_b()
get_percentage_true_b(dataset, verbose=True)

True B is the best candidate: 7595 / 10240 (74.17%)


np.float64(0.74169921875)

In [5]:
dataset.train()

[BEST 10% STD] True B is the best candidate: 797 / 1024 (77.83%)
[BEST 30% STD] True B is the best candidate: 2343 / 3072 (76.27%)
[BEST 60% STD] True B is the best candidate: 4640 / 6144 (75.52%)
[BEST 100% STD] True B is the best candidate: 7595 / 10240 (74.17%)


(False, None)

In [11]:
num_gen = dataset.params['num_gen']
n = dataset.params['n']
k = dataset.params['k']
q = dataset.params['q']

In [12]:
dataset.A.shape

(64, 16)

In [17]:
A_to_reduce = np.stack([dataset.A[ind] for ind in dataset.indices])
A_to_reduce[1]

array([[ -93.,  -98.,   30., -104.,  -20.,   40.,  -90.,  -38.,   60., -116.,  -25.,   56.,  -32.,  109.,   97.,   52.],
       [  38.,  -93.,  -98.,   30., -104.,  -20.,   40.,  -90.,  -52.,   60., -116.,  -25.,   56.,  -32.,  109.,   97.],
       [  90.,   38.,  -93.,  -98.,   30., -104.,  -20.,   40.,  -97.,  -52.,   60., -116.,  -25.,   56.,  -32.,  109.],
       [ -40.,   90.,   38.,  -93.,  -98.,   30., -104.,  -20., -109.,  -97.,  -52.,   60., -116.,  -25.,   56.,  -32.],
       [  20.,  -40.,   90.,   38.,  -93.,  -98.,   30., -104.,   32., -109.,  -97.,  -52.,   60., -116.,  -25.,   56.],
       [ 104.,   20.,  -40.,   90.,   38.,  -93.,  -98.,   30.,  -56.,   32., -109.,  -97.,  -52.,   60., -116.,  -25.],
       [ -30.,  104.,   20.,  -40.,   90.,   38.,  -93.,  -98.,   25.,  -56.,   32., -109.,  -97.,  -52.,   60., -116.],
       [  98.,  -30.,  104.,   20.,  -40.,   90.,   38.,  -93.,  116.,   25.,  -56.,   32., -109.,  -97.,  -52.,   60.],
       [ -72.,  -85., -116.,  -9

In [8]:
dataset.R[0][0]

array([[ 6., -2., -1.,  1., -1.,  5., -9., -1.,  0.,  4.,  1.,  8., -2.,  1.,  1.,  5., -3.,  1.,  3.,  0., -4., -5., -3.,  3.,  0., -1., -4., -2., -4.,  8., -4.,  8.,  2.,  5., -1., -2.,  2.,  3., -2.,  0.,  0.,  2.,  1., -4.,  3.,  5.,  0.,  0., -6., -4., -5., -4.,  0.,  0.,  0., -2., -1.,  3.,  3., -3.,  4., -4., -1.,  0., -2.,  1., -1.,  5., -4., -2.,  1.,  4., -4.,  3.,  1.,  1.,  0., -1., -2., -5.,  1., -2.,  5.,  5., -6.,  1.,  2.,  0.,  7., -3., -2.,  8.,  5., -1., -4.,  4.,  0.,  2.,  1.,  1.,  0.,  0.,  0.,  7.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
       [-5.,  6., -2., -1.,  1., -1.,  5., -9., -1.,  0.,  4.,  1.,  8., -2.,  1.,  1., -8., -3.,  1.,  3.,  0., -4., -5., -3.,  3.,  0., -1., -4., -2., -4.,  8., -4., -0.,  2.,  5., -1., -2.,  2.,  3., -2.,  0.,  0.,  2.,  1., -4.,  3.,  5.,  0., -0., -6., -4., -5., -4.,  0.,  0.,  0., -2., -1.,  3.,  3., -3.,  4., -4., -1.,  5., -2.,  1., -1.,  5.

In [24]:
mod_mult(dataset.R[0][0], A_to_reduce[0], q)

array([[ -86.,   18.,  -36., ...,  -76.,   -9.,  -35.],
       [  -9.,  -86.,   18., ...,   -2.,  -76.,   -9.],
       [-103.,   -9.,  -86., ...,  119.,   -2.,  -76.],
       ...,
       [ -21., -146.,    6., ...,   68.,  145., -121.],
       [  36.,  -21., -146., ...,   35.,   68.,  145.],
       [ -18.,   36.,  -21., ...,    9.,   35.,   68.]])

In [25]:
item = 3
parts = np.split(dataset.R[item][0], k)
parts = np.hstack([neg_circ(part).T for part in parts])

mod_mult(parts, A_to_reduce[item], q)

ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 64 is different from 16)

In [11]:
R_splitted = np.stack([np.hstack([neg_circ(part).T for part in np.split(dataset.R[i][0], k)]) for i in range(dataset.R.shape[0])])
mod_mult(R_splitted, A_to_reduce, q)

array([[[-26.,  18., -16., -36.,  12.,   2., -11.,  -7., -15., -34.,  -1., -18.,  -2.,  12.,  -3.,   6.],
        [  7., -26.,  18., -16., -36.,  12.,   2., -11.,  -6., -15., -34.,  -1., -18.,  -2.,  12.,  -3.],
        [ 11.,   7., -26.,  18., -16., -36.,  12.,   2.,   3.,  -6., -15., -34.,  -1., -18.,  -2.,  12.],
        [ -2.,  11.,   7., -26.,  18., -16., -36.,  12., -12.,   3.,  -6., -15., -34.,  -1., -18.,  -2.],
        [-12.,  -2.,  11.,   7., -26.,  18., -16., -36.,   2., -12.,   3.,  -6., -15., -34.,  -1., -18.],
        [ 36., -12.,  -2.,  11.,   7., -26.,  18., -16.,  18.,   2., -12.,   3.,  -6., -15., -34.,  -1.],
        [ 16.,  36., -12.,  -2.,  11.,   7., -26.,  18.,   1.,  18.,   2., -12.,   3.,  -6., -15., -34.],
        [-18.,  16.,  36., -12.,  -2.,  11.,   7., -26.,  34.,   1.,  18.,   2., -12.,   3.,  -6., -15.]],

       [[ -7.,  -7.,  16.,  24., -13.,  28.,  26., -27.,  25., -16.,  -3.,   6., -18.,  -6.,   0.,  26.],
        [ 27.,  -7.,  -7.,  16.,  24., -13.,

resampling R for more samples (not working)

In [36]:
dataset.R[0][1][:, n:]

array([[ -3.,   7.,  30., ..., -11.,   5., -10.],
       [ 10.,  -3.,   7., ...,  -3., -11.,   5.],
       [ -5.,  10.,  -3., ..., -16.,  -3., -11.],
       ...,
       [-25.,  10.,   2., ...,  -3.,   7.,  30.],
       [-30., -25.,  10., ...,  10.,  -3.,   7.],
       [ -7., -30., -25., ...,  -5.,  10.,  -3.]])

In [35]:
# Take two different R matrices for the same A_to_reduce matrix (e.g., index 0)
R1 = dataset.R[0][0]
R2 = dataset.R[0][1]

# Split each R into k parts
R1_parts = np.split(R1, k, axis=1)
R2_parts = np.split(R2, k, axis=1)

# Stack the split parts from both R1 and R2 along a new axis
combined_parts = np.hstack([R1_parts[0], R2_parts[1]])
combined_parts

array([[ -6.,   8.,  -8., ..., -11.,   5., -10.],
       [ 29.,  -6.,   8., ...,  -3., -11.,   5.],
       [ 19.,  29.,  -6., ..., -16.,  -3., -11.],
       ...,
       [ -3.,   5.,   3., ...,  -3.,   7.,  30.],
       [  8.,  -3.,   5., ...,  10.,  -3.,   7.],
       [ -8.,   8.,  -3., ...,  -5.,  10.,  -3.]])

In [40]:
mod_mult(combined_parts, A_to_reduce[0], q)

array([[  945., -1359.,  -296., ..., -1394., -1206.,   953.],
       [ -675.,   945., -1359., ..., -1203., -1394., -1206.],
       [ 1052.,  -675.,   945., ...,   591., -1203., -1394.],
       ...,
       [ -524.,  -778.,   334., ...,  1380.,  -339., -1225.],
       [  296.,  -524.,  -778., ...,  -953.,  1380.,  -339.],
       [ 1359.,   296.,  -524., ...,  1206.,  -953.,  1380.]])

From paper "Enhancing MLWE"

In [1]:
import numpy as np
from fractions import Fraction

import sys
import os

# Add the parent directory to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from ml_attack.utils import mod_mult
from ml_attack.lwe import neg_circ

from fpylll import IntegerMatrix, LLL

np.set_printoptions(linewidth=np.inf)

# Parameters
n = 8
k = 2
q = 3329
m = 10                # not divisible by n
h = m // n           # =1
g = m - h*n          # =2

dim_u_full = (h+1)*n # 8
dim_v = k*n          # 4
ambient_dim = dim_u_full + dim_v  # 12

print("Parameters:", dict(n=n, k=k, q=q, m=m, h=h, g=g,
                          ambient_dim=ambient_dim, rank_Lp=m+dim_v))

# Random A_full of shape ( (h+1)n x kn )
rng = np.random.default_rng(42)

# Create A_full using neg_circ for MLWE: each column is neg_circ of a random vector mod q
A_full = np.vstack([np.hstack([neg_circ(rng.integers(0, q, n)) for _ in range(k)]) for _ in range(h+1)])

A_used = A_full[:m, :]

print("\nA_full (8x4):\n", A_full)
print("\nA_used (first 6 rows):\n", A_used)

# --- Build ambient lattice basis B_full ---
I_u = np.eye(dim_u_full, dtype=int)
I_v = np.eye(dim_v, dtype=int)

top = np.hstack([I_u*4, A_full])
bottom = np.hstack([np.zeros((dim_u_full, dim_v), dtype=int), q*I_v])
B_full = np.vstack([top, bottom])

print("\nB_full (12x12) lattice basis:\n", B_full)

# --- Projection Π: zero out u-rows hn+g .. (h+1)n-1 ---
Pi = np.diag([0 if h*n + g <= r < (h+1)*n else 1 for r in range(ambient_dim)])
print("\nProjection Pi:\n", Pi)

# Apply projection
D_scaled = Pi @ B_full
print("\nD_scaled = Pi * B:\n", D_scaled)

D_int = IntegerMatrix.from_matrix(D_scaled.astype(int).tolist())
LLL.reduction(D_int)

D_reduced = np.zeros(D_scaled.shape)
D_int.to_matrix(D_reduced)
print("\nD_reduced (after LLL):\n", D_reduced)

# Remove the (n-g) zero vectors after reduction
# Remove all-zero rows
D_nonzero = D_reduced[~np.all(D_reduced == 0, axis=1)]
# Remove all-zero columns
#D_nonzero = D_nonzero[:, ~np.all(D_nonzero == 0, axis=0)]
print("\nD_nonzero (after removing zero rows):\n", D_nonzero)

print("D_nonzero shape:", D_nonzero.shape)
print("D_nonzero rank:", np.linalg.matrix_rank(D_nonzero))
print("m + n*k =", m + n * k)

Parameters: {'n': 8, 'k': 2, 'q': 3329, 'm': 10, 'h': 1, 'g': 2, 'ambient_dim': 32, 'rank_Lp': 26}

A_full (8x4):
 [[  297 -2321  -286 -2858 -1441 -1461 -2179 -2576   670 -2616 -2388 -2533 -2449 -3247 -1752  -313]
 [ 2576   297 -2321  -286 -2858 -1441 -1461 -2179   313   670 -2616 -2388 -2533 -2449 -3247 -1752]
 [ 2179  2576   297 -2321  -286 -2858 -1441 -1461  1752   313   670 -2616 -2388 -2533 -2449 -3247]
 [ 1461  2179  2576   297 -2321  -286 -2858 -1441  3247  1752   313   670 -2616 -2388 -2533 -2449]
 [ 1441  1461  2179  2576   297 -2321  -286 -2858  2449  3247  1752   313   670 -2616 -2388 -2533]
 [ 2858  1441  1461  2179  2576   297 -2321  -286  2533  2449  3247  1752   313   670 -2616 -2388]
 [  286  2858  1441  1461  2179  2576   297 -2321  2388  2533  2449  3247  1752   313   670 -2616]
 [ 2321   286  2858  1441  1461  2179  2576   297  2616  2388  2533  2449  3247  1752   313   670]
 [ 1708 -3085  -607 -1234 -1665 -1499 -2795  -426  2601  -756 -1499 -1476 -1815 -2738 -1339 -

In [25]:
R = D_nonzero[:, :m]
R.shape

(26, 10)

In [37]:
A_used

array([[  297, -2321,  -286, -2858, -1441, -1461, -2179, -2576,   670, -2616, -2388, -2533, -2449, -3247, -1752,  -313],
       [ 2576,   297, -2321,  -286, -2858, -1441, -1461, -2179,   313,   670, -2616, -2388, -2533, -2449, -3247, -1752],
       [ 2179,  2576,   297, -2321,  -286, -2858, -1441, -1461,  1752,   313,   670, -2616, -2388, -2533, -2449, -3247],
       [ 1461,  2179,  2576,   297, -2321,  -286, -2858, -1441,  3247,  1752,   313,   670, -2616, -2388, -2533, -2449],
       [ 1441,  1461,  2179,  2576,   297, -2321,  -286, -2858,  2449,  3247,  1752,   313,   670, -2616, -2388, -2533],
       [ 2858,  1441,  1461,  2179,  2576,   297, -2321,  -286,  2533,  2449,  3247,  1752,   313,   670, -2616, -2388],
       [  286,  2858,  1441,  1461,  2179,  2576,   297, -2321,  2388,  2533,  2449,  3247,  1752,   313,   670, -2616],
       [ 2321,   286,  2858,  1441,  1461,  2179,  2576,   297,  2616,  2388,  2533,  2449,  3247,  1752,   313,   670],
       [ 1708, -3085,  -607, -12

In [35]:
mod_mult(neg_circ(R[0][:8]).T, A_used[:8], q)

array([[ -485.,   851.,  1002., -1166.,  -634.,  1616.,   -23.,  1529.,  1052.,  1319., -1313.,   793.,   794.,  1136.,  -407.,  -167.],
       [-1529.,  -485.,   851.,  1002., -1166.,  -634.,  1616.,   -23.,   167.,  1052.,  1319., -1313.,   793.,   794.,  1136.,  -407.],
       [   23., -1529.,  -485.,   851.,  1002., -1166.,  -634.,  1616.,   407.,   167.,  1052.,  1319., -1313.,   793.,   794.,  1136.],
       [-1616.,    23., -1529.,  -485.,   851.,  1002., -1166.,  -634., -1136.,   407.,   167.,  1052.,  1319., -1313.,   793.,   794.],
       [  634., -1616.,    23., -1529.,  -485.,   851.,  1002., -1166.,  -794., -1136.,   407.,   167.,  1052.,  1319., -1313.,   793.],
       [ 1166.,   634., -1616.,    23., -1529.,  -485.,   851.,  1002.,  -793.,  -794., -1136.,   407.,   167.,  1052.,  1319., -1313.],
       [-1002.,  1166.,   634., -1616.,    23., -1529.,  -485.,   851.,  1313.,  -793.,  -794., -1136.,   407.,   167.,  1052.,  1319.],
       [ -851., -1002.,  1166.,   634., -

In [32]:
neg_circ(R[0][8:]).T

array([[-92., -44.],
       [ 44., -92.]])

In [27]:
nk = n * k
R_splitted = np.stack([np.hstack([neg_circ(part).T for part in np.array_split(row, range(nk, len(row), nk))]) for row in R])
R_splitted

array([[[  48.,   36.,    0.,  108.,  -48.,  -32.,   52.,   36.,  -92.,  -44.],
        [  44.,   48.,   36.,    0.,  108.,  -48.,  -32.,   52.,   36.,  -92.],
        [  92.,   44.,   48.,   36.,    0.,  108.,  -48.,  -32.,   52.,   36.],
        [ -36.,   92.,   44.,   48.,   36.,    0.,  108.,  -48.,  -32.,   52.],
        [ -52.,  -36.,   92.,   44.,   48.,   36.,    0.,  108.,  -48.,  -32.],
        [  32.,  -52.,  -36.,   92.,   44.,   48.,   36.,    0.,  108.,  -48.],
        [  48.,   32.,  -52.,  -36.,   92.,   44.,   48.,   36.,    0.,  108.],
        [-108.,   48.,   32.,  -52.,  -36.,   92.,   44.,   48.,   36.,    0.],
        [  -0., -108.,   48.,   32.,  -52.,  -36.,   92.,   44.,   48.,   36.],
        [ -36.,   -0., -108.,   48.,   32.,  -52.,  -36.,   92.,   44.,   48.]],

       [[ -12.,  -36.,  -52.,  -84.,  112.,   24.,  -60.,   12.,  120.,   16.],
        [ -16.,  -12.,  -36.,  -52.,  -84.,  112.,   24.,  -60.,   12.,  120.],
        [-120.,  -16.,  -12.,  -36.,  

In [28]:
mod_mult(R_splitted, A_used, q)

array([[[ 7.200e+01, -2.080e+02, -4.960e+02, -7.480e+02,  4.440e+02, -2.720e+02,  1.600e+02,  5.800e+02,  3.680e+02, -2.960e+02,  8.000e+01, -5.280e+02, -3.120e+02, -8.000e+00,  2.360e+02, -4.280e+02],
        [ 1.250e+03, -1.617e+03,  2.510e+02,  1.670e+02,  3.930e+02, -1.653e+03,  1.073e+03, -9.180e+02,  1.820e+02,  1.311e+03, -1.025e+03,  6.630e+02, -1.502e+03, -8.010e+02,  1.417e+03,  6.190e+02],
        [ 1.490e+02, -1.458e+03, -4.540e+02, -2.770e+02, -2.160e+02,  1.342e+03, -1.413e+03, -6.200e+02,  1.427e+03,  1.036e+03, -1.451e+03,  5.850e+02,  1.623e+03, -1.542e+03, -3.340e+02, -8.490e+02],
        [ 1.585e+03, -1.656e+03, -7.710e+02,  1.551e+03, -2.650e+02, -1.558e+03, -7.160e+02,  1.663e+03,  2.900e+02, -1.430e+02,  1.237e+03, -1.105e+03, -4.750e+02,  3.100e+02, -5.900e+02, -1.659e+03],
        [ 9.750e+02,  8.100e+01,  5.540e+02, -1.327e+03,  8.020e+02, -1.010e+03,  9.830e+02, -9.230e+02, -1.550e+02,  3.180e+02, -1.080e+03, -1.493e+03, -6.040e+02, -2.390e+02,  1.262e+03, -3.

In [19]:
import numpy as np
from typing import List, Tuple, Dict, Any
from tqdm import tqdm

def build_matrices_from_blocks(
    n: int,
    m: int,
    num_blocks: int,
    num_matrices: int,
    seed: int = None,
    verbose: bool = False
) -> Tuple[List[np.ndarray], List[Dict[str, Any]]]:
    """
    Build `num_matrices` matrices each with exactly `m` rows from `blocks`.

    - Always uses a global ordering (queue of block indices).
    - Queue is a random permutation of all blocks; once exhausted, it is refilled with a new random permutation.
    - If num_matrices >= num_blocks: ensures each block is the *first* block of a different matrix.
    - Avoids using the same block twice in a matrix until all blocks have been used there.
    """

    rng = np.random.default_rng(seed)

    matrices_segments = [[] for _ in range(num_matrices)]
    used_blocks = {mi: [] for mi in range(num_matrices)}  # track how many times each block was used

    # --- Coverage offset strategy ---
    coverage_offsets = {bidx: [i * m for i in range((n + m - 1) // m)] for bidx in range(num_blocks)}
    coverage_counters = {bidx: 0 for bidx in range(num_blocks)}
    used_offsets = {bidx: set() for bidx in range(num_blocks)}  # track all chosen offsets

    def choose_offset(bidx: int) -> int:
        """Pick next systematic offset if available, else a fresh random offset (avoid repeats)."""
        cnt = coverage_counters[bidx]
        if cnt < len(coverage_offsets[bidx]):
            # Use systematic coverage offset
            offset = coverage_offsets[bidx][cnt]
            coverage_counters[bidx] += 1
        else:
            # Random but prefer unused offsets
            all_offsets = set(range(n))
            candidates = list(all_offsets - used_offsets[bidx])
            if candidates:
                offset = int(rng.choice(candidates))
            else:
                offset = int(rng.integers(0, n))
        used_offsets[bidx].add(offset)
        return offset
        
    # --- Queue of blocks (reshuffled each epoch) ---
    def next_block(queue: list) -> int:
        if not queue:
            new_epoch = rng.permutation(num_blocks).tolist()
            queue.extend(new_epoch)
        return queue.pop(0)

    # Initialize queue (first epoch)
    queue = rng.permutation(num_blocks).tolist()

    # --- Step 1: Assign first block for coverage ---
    if num_matrices >= num_blocks:
        # one block as first in each distinct matrix
        matrix_indices = rng.choice(num_matrices, size=num_blocks, replace=False).tolist()
        for mi, bidx in zip(matrix_indices, rng.permutation(num_blocks)):
            start = choose_offset(bidx)
            take = min(n, m)
            rotated = ((np.arange(n) - start) % n)  + bidx * n
            matrices_segments[mi].append(rotated[:take])
            used_blocks[mi].append(int(bidx))
    else:
        # distribute blocks round-robin
        for i, bidx in enumerate(rng.permutation(num_blocks)):
            target_mi = i % num_matrices
            start = choose_offset(bidx)
            current_rows = sum(seg.shape[0] for seg in matrices_segments[target_mi])
            take = min(n, m - current_rows)
            if take > 0:
                rotated = ((np.arange(n) - start) % n) + bidx * n
                matrices_segments[target_mi].append(rotated[:take])
                used_blocks[target_mi].append(int(bidx))

    # --- Step 2: Fill each matrix up to m rows ---
    for mi in range(num_matrices):
        current_rows = sum(seg.shape[0] for seg in matrices_segments[mi])
        current_blocks = set(used_blocks[mi])

        while current_rows < m:
            bidx = next_block(queue)
            # If already used in this matrix AND there are still unused blocks left,
            # put it back at the end of the queue and try another, unless this is the last block remaining in the queue
            if bidx in current_blocks and len(current_blocks) < num_blocks:
              # Check if there are any blocks in the queue that have not been used in this matrix
              unused_in_queue = [idx for idx in queue if idx not in current_blocks]
              if unused_in_queue:
                queue.append(bidx)
                continue

            start = choose_offset(bidx)
            need = m - current_rows
            take = min(n, need)
            rotated = ((np.arange(n) - start) % n) + bidx * n

            matrices_segments[mi].append(rotated[:take])
            used_blocks[mi].append(int(bidx))
            current_blocks.add(bidx)
            current_rows += take

    # --- Finalize matrices ---
    matrices = np.stack([np.hstack(segs)[:m] for segs in matrices_segments])

    if verbose:
        # Compute overall coverage as the fraction of unique entries in all matrices
        all_entries = np.concatenate(matrices)
        coverage = len(set(all_entries.flatten())) / (num_blocks * n)
        print(f"Overall coverage: {coverage:.4f}")

        # Compute overall reuse: average number of times each entry appears in all matrices
        unique_entries, counts = np.unique(all_entries, return_counts=True)
        overall_reuse = counts.mean()
        print(f"Overall reuse (mean count per unique entry): {overall_reuse:.4f}")

    return matrices

def uniqueness_stats(matrices):
    ordered = len({M.tobytes() for M in matrices})
    unordered = len({
        tuple(map(tuple, np.atleast_2d(M)[np.lexsort(np.atleast_2d(M).T[::-1])]))
        for M in matrices
    })
    return ordered, unordered

# --- Demo ---
if __name__ == "__main__":
    # toy blocks: 4 blocks, block size n=8
    n = 256
    k = 2
    num_gen = 4
    num_blocks = num_gen * k

    #blocks = [np.array([[b, r] for r in range(n)]) for b in range(num_blocks)]
    m = 1000
    num_matrices = 1000
    min_matrices = (n // (m + 1) + 1) * num_gen * k
    m_min = int(np.ceil(n / (num_matrices // (num_gen * k))))

    print(f"num_blocks={num_blocks}, m={m}, num_matrices={num_matrices}, min_matrices={min_matrices}, m_min={m_min}")

    seen_md = set()
    mats = build_matrices_from_blocks(n, m, num_blocks, num_matrices, seed=3, verbose=True)
    print(mats)

    # --- Stress test: repeat 1000 times to check for infinite loop or stack ---
    for trial in tqdm(range(10000)):
        mats_test = build_matrices_from_blocks(n, m, num_blocks, num_matrices, seed=trial)
        ordered, unordered = uniqueness_stats(mats_test)
        assert ordered == len(mats_test), f"Ordered uniqueness failed at trial {trial}"
        assert unordered == len(mats_test), f"Unordered uniqueness failed at trial {trial}"
    print("Stress test passed: no infinite loop or stack after 1000 runs.")


num_blocks=8, m=1000, num_matrices=1000, min_matrices=8, m_min=3
Overall coverage: 1.0000
Overall reuse (mean count per unique entry): 488.2812
[[1626 1627 1628 ...  410  411  412]
 [1040 1041 1042 ...  236  237  238]
 [ 107  108  109 ...  947  948  949]
 ...
 [ 312  313  314 ... 1473 1474 1475]
 [ 720  721  722 ... 1524 1525 1526]
 [  66   67   68 ... 1841 1842 1843]]


  0%|          | 21/10000 [00:11<1:27:35,  1.90it/s]


KeyboardInterrupt: 